### Step1: Import Libraries and API Keys

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
from openai import OpenAI
import gradio as gr
import json
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

print(OPENAI_API_KEY[:8])

### Step 2: Simple RAG with Guardrails & Dynamic context Injection

In [ ]:
system_message = """You are a digital twin of Mohammed Imran Ali. When people talk to you, you respond as Imran -  in first person, using his voice,personality, and knowledge.
Important: do not make things up. If you don't know an answer, say you don't know.
The only factual information available to you is what's in this system message.
You cannot get any more facts about Imran from the internet or make them up.
***

Here's the information about Imran to help you embody him:
Mohammed Imran Ali is an experienced IT Service Desk and IT Operations Leader based in Hyderabad, Telangana, India, with over 11+ years of experience in enterprise IT support, cloud exposure, and service delivery excellence. Currently working as an Assistant Manager – IT Service Desk at Cotiviti since January 2019, I lead and manage a team of 30+ service desk professionals supporting global enterprise users across voice and chat channels.

In his current role (01/2019 Present), He oversee end-to-end IT service desk operations, ensuring SLA compliance, incident resolution efficiency, and continuous service improvement. I have successfully implemented ITIL best practices, optimized workforce planning, and introduced automation in ticketing workflows using tools like Jira Service Desk and ServiceNow. My responsibilities also include Microsoft 365 administration, stakeholder collaboration, performance management, and driving operational excellence across IT infrastructure support.

Prior to this, He worked with Genpact as a Process Developer from September 2014 to September 2018. During this period, I specialized in L2/L3 Identity and Access Management (IAM) support, handling user lifecycle management including provisioning, de-provisioning, password resets, and access control across platforms like Active Directory, Office 365, Citrix, and RDP environments. I also contributed to improving ServiceNow ITSM workflows and maintaining knowledge-centered support documentation aligned with best practices.

Education
He hold a Post Graduate Diploma in Cyber Crime from University of Hyderabad (April 2013 November 2015) and a Bachelor of Computer Science from Osmania University (July 2007 – December 2012), which built a strong academic foundation for His career in IT and security.

Languages
He is proficient in:
English Full Professional Proficiency
French Elementary Proficiency
Certifications & Technical Strengths

He is a certified in ITIL V3, Lean Six Sigma Green Belt, and AWS Cloud Architecture, with hands-on exposure to AWS services such as EC2 and S3. My expertise spans IT service management tools (ServiceNow, Jira), Microsoft 365 administration, automation, and cloud-based infrastructure.

Personal Strength & Value Proposition

What truly defines his ability to lead with impact while continuously evolving with technology. He brings a strong mix of people leadership, process optimization, and technical acumen, enabling organizations to achieve higher efficiency, improved user experience, and scalable IT operations.

He is passionate about driving digital transformation, exploring AI-driven service desk innovations, and building high-performing teams that consistently deliver beyond expectations. His approach is rooted in ownership, continuous learning, and a commitment to excellence, making me a reliable and forward-thinking IT leader.
***
"""


In [ ]:
Topic_Context = {
    "training_itil": "***Imran joined ITILv3 certification training in September 2018 and January 2019.***",
    "guava": "***Imran enjoys guava juice.***",
    "snack": "***Imran enjoys snacks with some sweet added guava.***",
    "brinjal": "***Imran's least liked vegetable is brinjal, but he likes it air fried.***"
}
print(Topic_Context["training_itil"])

###Step 3: Prepare the list of tools for the LLM

In [ ]:
tools = []

###Step3a : Add Too-calling functionality (Pushover)

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

#Test Pushover
import requests

def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)


    #Describe Pushover as an LLM tool

send_notification_function = {
    "name": "send_notification",
    "description": "Send one notification. Call this two times if multiple notifications are needed.",
    "parameters":{
        "type": "object",
        "properties":{
            "message":{
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
}
}

tools.append({"type":"function", "function":send_notification_function})

In [ ]:
#Test notification
#send_notification("Hello to myselft, from this AI engineering training")

###Step 3b: Add dice-rolling functionality (Pushover)

In [ ]:
import random
#simulates rolling a single six-sided die
def dice_roll():
    result = random.randint(1,6)
    return  result

#Describe fucntion for the LLM
roll_dice_function = {
      "name": "dice_roll",
      "description": "Simulates rolling a single six-sided die and returns the result. Use this when the user wants to roll a die for games, or random number generation.",
      "parameters":{
         "type": "object",
         "properties":{},
         "required": []
    }
}


#Add fucntion to list of tools of LLM
tools.append({"type":"function", "function":roll_dice_function})


###Step 4: function to Handle LLM Tool calls

In [ ]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        #print(f" Calling function {function_name}")

        #Route to the appropriate function based on function_name
        if function_name == "send_notification":
             #tool_call = tool_calls[0] #assuming just one tool call
            send_notification(args["message"])
            content = f"Notification sent: {args['message']}"
        elif function_name == "dice_roll":
            content = f"Rolled: {dice_roll()}"
         #elif function_name == "insert_function_name_3":
        #    content = insert_function_name_3(args["messag"])
        #...
        else:
            content = f"Unknown function: {function_name}"




           #Actually send the notification, i.e, call the tool
        
            #print(f"Sent notification: {args['message']}")

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
    }

        tool_results.append(tool_call_result)

    return tool_results   


#what to add to our "context"(about tool call), a dictionary.

###Step 5: Function to process the Conversation Turn

In [ ]:


def respond_ai(message, history):
      #Inject Dynamic Context based on keywords in the message
    system_message_enhanced = system_message
    for keyword, context in Topic_Context.items():
        if keyword in message.lower():
            system_message_enhanced += "\n\n" + context
          
    #As usual

    messages = [{"role": "system", "content": system_message_enhanced}] + history + [{"role": "user", "content": message}]
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools = tools
    )

    message = response.choices[0].message

    #Check if model wants to call a tool

    while message.tool_calls:
        from pprint import pprint
        pprint (message.tool_calls)

        tool_result = handle_tool_call(message.tool_calls)
        messages.append(message)
        messages.extend(tool_result)

        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tools
        )
        message = response.choices[0].message


    return(message.content)






###STEP 6: GRADIO LAUNCH

In [ ]:
gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)